## Interpretación del Modelo Seleccionado - Dataset Boston Housing

### By:
Rafael Garcia

### Date:
2026-08-20

### Description:

Este notebook corresponde a la **Tarea 7: Interpretación del modelo seleccionado** del Proyecto 1 (Dataset Estático) del curso de MLOps.

El objetivo es analizar las características del modelo obtenido en la Tarea 6, entender sus fortalezas y debilidades, e identificar oportunidades de mejora. Se incluye:

- Interpretación del modelo y sus atributos más importantes
- Análisis de Learning Curve y escalabilidad
- Consecuencias de las malas predicciones
- Tipos de errores que comete el modelo
- Causas de los errores

## 📚 Import libraries

In [ ]:
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import learning_curve, train_test_split

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)

## 💾 Load data and model

In [ ]:
df = pd.read_parquet("../../data/02_intermediate/boston_housing_clean.parquet")

TARGET = "medv"
ID_COL = "ID"
TEST_SIZE = 0.2
RANDOM_STATE = 42
N_FOLDS = 10

df = df.drop(columns=[ID_COL])
df = df.dropna(subset=[TARGET])

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

for dataset in [X_train, X_test]:
    dataset["chas"] = dataset["chas"].astype(str)
    dataset["rad"] = dataset["rad"].astype(float)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# Cargar el mejor modelo de la Tarea 6
best_model = joblib.load("../../models/best_model.joblib")
print("Modelo cargado:")
for step_name, step in best_model.named_steps.items():
    print(f"  {step_name}: {type(step).__name__}")

# Obtener nombre del modelo
model_name = type(best_model.named_steps["model"]).__name__

In [ ]:
# Entrenar y generar predicciones
best_model.fit(X_train, y_train)
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

print(f"RMSE Train: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
print(f"RMSE Test:  {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"R² Train:   {r2_score(y_train, y_pred_train):.4f}")
print(f"R² Test:    {r2_score(y_test, y_pred_test):.4f}")

## 🔍 Interpretación del modelo

### Atributos más importantes

Se utiliza **Permutation Importance** para medir la importancia de cada feature. Este método es agnóstico al modelo y mide cuánto empeora el score al permutar aleatoriamente cada feature.

In [ ]:
# Permutation Importance en test set
perm_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
)

feature_names = X_test.columns.tolist()
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance_mean": perm_result.importances_mean,
    "Importance_std": perm_result.importances_std,
}).sort_values("Importance_mean", ascending=False)

print("=== Permutation Importance (Test Set) ===")
importance_df.round(4)

In [ ]:
# Gráfico de importancia
fig, ax = plt.subplots(figsize=(10, 6))

sorted_idx = importance_df["Importance_mean"].values.argsort()
ax.barh(
    importance_df["Feature"].values[sorted_idx],
    importance_df["Importance_mean"].values[sorted_idx],
    xerr=importance_df["Importance_std"].values[sorted_idx],
    color="steelblue",
    edgecolor="black",
    alpha=0.7,
    capsize=3,
)
ax.set_xlabel("Importancia (Permutation Importance)")
ax.set_title(f"Importancia de Features - {model_name}")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
# Interpretación de las features más importantes
top_features = importance_df.head(5)
print("=== Top 5 Features más importantes ===")
print()

FEATURE_DESCRIPTIONS = {
    "lstat": "% población de estatus bajo -> a mayor lstat, menor precio",
    "rm": "Número de habitaciones -> a más habitaciones, mayor precio",
    "dis": "Distancia a centros de empleo -> mayor cercanía, mayor precio",
    "crim": "Tasa de criminalidad -> mayor crimen, menor precio",
    "nox": "Concentración de contaminantes -> más contaminación, menor precio",
    "ptratio": "Ratio alumnos/profesor -> mayor ratio, menor precio",
    "tax": "Tasa de impuestos -> mayor impuesto, menor precio",
    "age": "Antigüedad viviendas -> más antiguas, menor precio",
    "indus": "Proporción zona industrial -> más industria, menor precio",
    "black": "Composición demográfica -> variable éticamente cuestionable",
    "zn": "Proporción terreno residencial",
    "chas": "Colindante con río Charles -> efecto premium",
    "rad": "Accesibilidad a autopistas",
}

for _, row in top_features.iterrows():
    feat = row["Feature"]
    desc = FEATURE_DESCRIPTIONS.get(feat, "")
    print(f"  {feat}: importancia = {row['Importance_mean']:.4f} ± {row['Importance_std']:.4f}")
    print(f"    Interpretación: {desc}")
    print()

## 📈 Análisis de Learning Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

train_sizes = np.linspace(0.1, 1.0, 10)
train_sizes_abs, train_scores, test_scores = learning_curve(
    best_model,
    X_train,
    y_train,
    cv=N_FOLDS,
    train_sizes=train_sizes,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

train_scores = -train_scores
test_scores = -test_scores

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

ax.fill_between(
    train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.1, color="blue"
)
ax.fill_between(
    train_sizes_abs, test_mean - test_std, test_mean + test_std, alpha=0.1, color="orange"
)
ax.plot(train_sizes_abs, train_mean, "o-", color="blue", label="Train RMSE")
ax.plot(train_sizes_abs, test_mean, "o-", color="orange", label="Validation RMSE")
ax.set_xlabel("Tamaño del conjunto de entrenamiento")
ax.set_ylabel("RMSE")
ax.set_title(f"Learning Curve - {model_name}")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== Interpretación de la Learning Curve ===")
gap = test_mean[-1] - train_mean[-1]
print(f"Train RMSE final: {train_mean[-1]:.4f}")
print(f"Validation RMSE final: {test_mean[-1]:.4f}")
print(f"Gap: {gap:.4f}")
if gap > 2:
    print("Diagnóstico: Overfitting - el modelo memoriza pero no generaliza bien.")
    print("Recomendación: más datos, regularización, o modelo más simple.")
elif test_mean[-1] > 6:
    print("Diagnóstico: Underfitting - el modelo no captura la complejidad de los datos.")
    print("Recomendación: modelo más complejo, más features, o menos regularización.")
else:
    print("Diagnóstico: Buen equilibrio entre bias y varianza.")

## ⏱️ Gráficas de escalabilidad

In [ ]:
fractions = np.linspace(0.1, 1.0, 10)
times_list = []
scores_list = []
sizes_list = []

for frac in fractions:
    n = int(len(X_train) * frac)
    X_sub = X_train.iloc[:n]
    y_sub = y_train.iloc[:n]

    start = time.time()
    best_model.fit(X_sub, y_sub)
    elapsed = time.time() - start

    y_pred = best_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    times_list.append(elapsed)
    scores_list.append(rmse)
    sizes_list.append(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sizes_list, times_list, "o-", color="steelblue")
axes[0].set_xlabel("Tamaño del dataset")
axes[0].set_ylabel("Tiempo de entrenamiento (s)")
axes[0].set_title(f"Escalabilidad: Tiempo - {model_name}")
axes[0].grid(True, alpha=0.3)

axes[1].plot(sizes_list, scores_list, "o-", color="coral")
axes[1].set_xlabel("Tamaño del dataset")
axes[1].set_ylabel("RMSE (Test)")
axes[1].set_title(f"Escalabilidad: Score - {model_name}")
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"Análisis de Escalabilidad - {model_name}", fontsize=14)
plt.tight_layout()
plt.show()

## ⚠️ Consecuencias de las malas predicciones

En el contexto de predicción de precios de viviendas, las malas predicciones tienen consecuencias diferentes según la dirección del error:

In [ ]:
# Reentrenar con todo el train set
best_model.fit(X_train, y_train)
y_pred_test = best_model.predict(X_test)
residuos = y_test.values - y_pred_test

sobreestimados = (residuos < 0).sum()
subestimados = (residuos > 0).sum()

print("=== Consecuencias de las Malas Predicciones ===")
print()
print(f"Predicciones que sobreestiman el precio: {sobreestimados} ({sobreestimados / len(residuos) * 100:.1f}%)")
print("  Consecuencia: el comprador paga de más, el vendedor gana de más.")
print("  Riesgo: pérdida financiera para compradores, préstamos hipotecarios sobrevaluados.")
print()
print(f"Predicciones que subestiman el precio: {subestimados} ({subestimados / len(residuos) * 100:.1f}%)")
print("  Consecuencia: el vendedor pierde dinero, el comprador obtiene una ganga.")
print("  Riesgo: los vendedores pueden rechazar la estimación y perder confianza en el modelo.")
print()

LARGE_ERROR_THRESHOLD = 5
large_errors = (np.abs(residuos) > LARGE_ERROR_THRESHOLD).sum()
print(f"Errores grandes (> ${LARGE_ERROR_THRESHOLD}k): {large_errors} ({large_errors / len(residuos) * 100:.1f}%)")
print("  Un error > $5,000 puede ser inaceptable en una transacción real.")

## 🔎 ¿Qué tipo de errores comete el modelo?

In [ ]:
# Análisis de errores por segmento de precio
PRICE_BINS = [0, 15, 25, 35, 51]
PRICE_LABELS = ["Bajo (<15k)", "Medio (15-25k)", "Alto (25-35k)", "Premium (>35k)"]

test_analysis = pd.DataFrame({
    "real": y_test.values,
    "pred": y_pred_test,
    "error": residuos,
    "abs_error": np.abs(residuos),
})
test_analysis["segmento"] = pd.cut(
    test_analysis["real"], bins=PRICE_BINS, labels=PRICE_LABELS
)

print("=== Errores por Segmento de Precio ===")
segment_errors = test_analysis.groupby("segmento", observed=True).agg(
    n=("error", "count"),
    error_medio=("error", "mean"),
    mae=("abs_error", "mean"),
    rmse=("error", lambda x: np.sqrt((x**2).mean())),
    error_max=("abs_error", "max"),
).round(4)
segment_errors

In [ ]:
# Visualización de errores por segmento
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuos vs valor real
scatter = axes[0].scatter(
    test_analysis["real"],
    test_analysis["error"],
    c=test_analysis["segmento"].cat.codes,
    cmap="viridis",
    alpha=0.6,
    s=30,
)
axes[0].axhline(y=0, color="r", linestyle="--")
axes[0].set_xlabel("Valor real (medv)")
axes[0].set_ylabel("Error (real - predicción)")
axes[0].set_title("Residuos por Valor Real")

# Distribución de errores
axes[1].hist(residuos, bins=25, edgecolor="black", alpha=0.7, color="steelblue")
axes[1].axvline(x=0, color="r", linestyle="--")
axes[1].set_xlabel("Error (real - predicción)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de Errores")

# Predicción vs real
axes[2].scatter(y_test, y_pred_test, alpha=0.5, s=20, color="steelblue")
axes[2].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    label="Predicción perfecta",
)
axes[2].set_xlabel("Valor real")
axes[2].set_ylabel("Predicción")
axes[2].set_title(f"Predicción vs Real (R²={r2_score(y_test, y_pred_test):.4f})")
axes[2].legend()

plt.suptitle(f"Análisis de Errores - {model_name}", fontsize=14)
plt.tight_layout()
plt.show()

## 🔬 ¿A qué se deben los errores?

In [ ]:
# Identificar los peores errores
WORST_N = 10
test_full = X_test.copy()
test_full["real"] = y_test.values
test_full["pred"] = y_pred_test
test_full["error"] = residuos
test_full["abs_error"] = np.abs(residuos)

worst = test_full.nlargest(WORST_N, "abs_error")
print(f"=== {WORST_N} Peores Predicciones ===")
worst[["real", "pred", "error", "rm", "lstat", "crim", "nox"]].round(4)

In [ ]:
# Análisis de causas de error
print("=== Análisis de Causas de Error ===")
print()

# 1. Outliers
MEDV_CEILING = 50
ceiling_count = (y_test == MEDV_CEILING).sum()
print(f"1. OUTLIERS Y CENSURA:")
print(f"   Registros con medv = {MEDV_CEILING} (techo): {ceiling_count}")
if ceiling_count > 0:
    ceiling_errors = test_full[test_full["real"] == MEDV_CEILING]["abs_error"]
    print(f"   Error promedio en esos registros: {ceiling_errors.mean():.4f}")
    print("   Causa: el dataset tiene un techo artificial en medv=50.")
    print("   Impacto: el modelo no puede predecir precios reales > $50k.")
print()

# 2. Valores extremos en features
print("2. VALORES EXTREMOS EN FEATURES:")
for feat in ["crim", "rm", "lstat"]:
    feat_vals = test_full[feat].astype(float)
    q1 = feat_vals.quantile(0.25)
    q3 = feat_vals.quantile(0.75)
    iqr = q3 - q1
    outlier_mask = (feat_vals < q1 - 1.5 * iqr) | (feat_vals > q3 + 1.5 * iqr)
    if outlier_mask.sum() > 0:
        outlier_error = test_full.loc[outlier_mask, "abs_error"].mean()
        normal_error = test_full.loc[~outlier_mask, "abs_error"].mean()
        print(f"   {feat}: {outlier_mask.sum()} outliers, error={outlier_error:.2f} vs normal={normal_error:.2f}")
print()

# 3. Efecto de la variable black
print("3. CONSIDERACIÓN ÉTICA - VARIABLE BLACK:")
black_vals = test_full["black"].astype(float)
BLACK_THRESHOLD = 300
low_black = black_vals < BLACK_THRESHOLD
if low_black.sum() > 0:
    error_low = test_full.loc[low_black, "abs_error"].mean()
    error_high = test_full.loc[~low_black, "abs_error"].mean()
    print(f"   Registros con black < {BLACK_THRESHOLD}: {low_black.sum()}, error promedio: {error_low:.4f}")
    print(f"   Registros con black >= {BLACK_THRESHOLD}: {(~low_black).sum()}, error promedio: {error_high:.4f}")
    print("   Nota: esta variable codifica composición racial y debería evaluarse")
    print("   si su inclusión introduce sesgos discriminatorios en el modelo.")
print()

# 4. Errores por encoders
print("4. ENCODERS:")
print("   chas (OneHotEncoder): variable binaria, encoding simple, no es fuente de error.")
print("   rad (StandardScaler): variable ordinal tratada como numérica, podría")
print("   beneficiarse de un encoding ordinal explícito en futuras iteraciones.")

## 💾 Guardar modelo final

In [ ]:
# El modelo ya está guardado en models/best_model.joblib desde la Tarea 6
# Verificar que sigue funcionando
model_path = "../../models/best_model.joblib"
model_loaded = joblib.load(model_path)
y_verify = model_loaded.predict(X_test)

print(f"Modelo verificado: {model_path}")
print(f"Predicciones consistentes: {np.allclose(y_pred_test, y_verify)}")
print(f"\nModelo listo para la demo funcional (Tarea 8).")

## 📊 Analysis of Results and Conclusions

### Interpretación de resultados

1. **Features más importantes**: consistente con lo encontrado en el EDA (Tarea 3), `lstat` y `rm` son los predictores dominantes. Esto valida que el feature engineering (Tarea 4) fue efectivo al aplicar transformación logarítmica a `lstat`.

2. **Learning Curve**: muestra el equilibrio entre bias y varianza del modelo. El gap entre train y validation indica el nivel de overfitting.

3. **Tipos de errores**:
   - El modelo tiene mayor dificultad prediciendo viviendas premium (>$35k), donde los errores son más grandes.
   - Los registros con medv=50 (techo del dataset) generan errores sistemáticos por la censura de datos.
   - Los outliers en `crim` y `rm` producen errores más grandes que los registros normales.

4. **Causas de los errores**:
   - **Censura de datos**: el techo en medv=50 limita la capacidad del modelo en el segmento premium.
   - **Outliers**: valores extremos en features producen predicciones menos confiables.
   - **Complejidad del problema**: el precio de una vivienda depende de factores no capturados en el dataset (estado de la propiedad, renovaciones, mercado local).

5. **Consideración ética**: la variable `black` captura información demográfica racial. Su inclusión podría introducir sesgos discriminatorios. Se recomienda evaluar el impacto de excluirla en futuras iteraciones.

### Conclusiones

- El modelo seleccionado es robusto y supera significativamente al baseline.
- Las principales fuentes de error son la censura de datos y los outliers, no deficiencias del modelo.
- El pipeline completo (preprocessor + modelo) está listo para desplegarse en una demo funcional.

## 💡 Proposals and Ideas

### Experimentos para la siguiente iteración

1. **Eliminar registros con medv=50**: evaluar si quitar los datos censurados mejora la predicción en el rango real de precios.
2. **Excluir la variable `black`**: medir el impacto en rendimiento y documentar la decisión ética.
3. **Tratamiento de outliers en `crim`**: aplicar winsorización o capping para reducir el impacto de valores extremos.
4. **Encoding alternativo para `rad`**: probar OrdinalEncoder en lugar de tratar como numérica continua.
5. **Agregar interacciones**: incluir `rm²` y `rm × lstat` como features derivadas.

### Tarea 8 (Demo funcional)

- Crear una interfaz web con Streamlit o Gradio donde el usuario ingrese los valores de las features y el modelo devuelva la predicción del precio.

## 📖 References

- Guía del proyecto (Tarea 7): [joserzapata.github.io/post/ciencia-datos-proyecto-python/7-model_interpretation/](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/7-model_interpretation/)
- Scikit-learn Permutation Importance: [sklearn.inspection.permutation_importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- Christoph Molnar, *Interpretable Machine Learning*: [christophm.github.io/interpretable-ml-book/](https://christophm.github.io/interpretable-ml-book/)